# smtpBERT - URL Phishing Detection Training

This notebook trains the DistilBERT model for SMTP phishing detection.

**Runtime:** Runtime → Change runtime type → GPU T4

**Duration:** ~15-30 min for quick test, 2-4 hours for full training

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Upload Project Files

**Option A:** Upload `smtpBERT.zip` to Google Drive, then run cell below
**Option B:** Upload directly to Colab (Files sidebar → Upload)

In [ ]:
# Copy and extract project from Drive
!cp /content/drive/MyDrive/smtpBERT.zip /content/ 2>/dev/null || echo 'File not in Drive, upload manually or check path'
!unzip -o smtpBERT.zip -d /content/ -q
!ls /content/

## 3. Install Dependencies

In [ ]:
# Install PyTorch with CUDA support
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q

# Install all other dependencies
!pip install transformers datasets accelerate scikit-learn pandas numpy matplotlib streamlit aiosmtpd requests psutil tqdm xgboost joblib -q

print('Dependencies installed!')

## 4. Verify GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('WARNING: Running on CPU - training will be slow')

## 5. Configure Training Parameters

In [ ]:
import os
import sys

# Change to project directory
os.chdir('/content')
sys.path.insert(0, '/content')

# ==== TRAINING CONFIGURATION ====
# Adjust these values for your training

NUM_EPOCHS = 3          # Recommended: 3-10 for good accuracy
BATCH_SIZE = 16         # Adjust based on GPU memory (16-32 for T4)
LEARNING_RATE = 2e-5    # Standard for BERT fine-tuning
WEIGHT_DECAY = 0.01
MAX_TRAIN_SAMPLES = 100000  # Use None for full dataset, or number to limit

# Apply to environment
os.environ['NUM_EPOCHS'] = str(NUM_EPOCHS)
os.environ['BATCH_SIZE'] = str(BATCH_SIZE)
os.environ['LEARNING_RATE'] = str(LEARNING_RATE)

print(f'Configuration:')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Batch Size: {BATCH_SIZE}')
print(f'  Learning Rate: {LEARNING_RATE}')
print(f'  Max Train Samples: {MAX_TRAIN_SAMPLES if MAX_TRAIN_SAMPLES else "All"}')

## 6. Run Training

In [ ]:
# Import and run training
from src.models.distilbert_trainer import train_model

# Quick test (1 epoch, 5k samples) - recommended first run
# os.environ['NUM_EPOCHS'] = '1'

print('Starting training...')
print('='*60)

metrics = train_model()

print('='*60)
print('Training complete!')

## 7. Test Model

In [ ]:
# Test the trained model
from src.models.distilbert_trainer import test_model

test_model()

## 8. Download Trained Model

In [ ]:
# Zip the trained model
!cd /content && zip -r phishing_model.zip phishing_model/ -q

# Copy to Drive
!cp /content/phishing_model.zip /content/drive/MyDrive/

print('Model saved to /content/drive/MyDrive/phishing_model.zip')
print('\nTo use in local project:')
print('1. Download from Drive')
print('2. Extract to D:\\smtpBERT\\phishing_model\\')

---

## Quick Reference

| Task | Command |
|------|---------|
| Test only (no training) | `from src.models.distilbert_trainer import test_model; test_model()` |
| Train with custom epochs | `os.environ['NUM_EPOCHS'] = '5'; train_model()` |
| View GPU memory | `!nvidia-smi` |
| Check training logs | `!cat /content/logs/*.log` |